# Customer Churn - Exploratory Data Analysis

This notebook performs data cleaning, churn rate analysis, and visual exploration of the customer dataset.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Load dataset
DATA_PATH = os.path.join("..", "data", "customers.csv")
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Data Overview & Cleaning

In [ ]:
# Basic info
print("=" * 50)
print("DATASET INFO")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found.")

# Check for duplicates
dup_count = df.duplicated(subset="customer_id").sum()
print(f"\nDuplicate customer_ids: {dup_count}")

# Data types and unique values for categoricals
print("\nCategorical value counts:")
for col in ["contract_type", "payment_method", "churn"]:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
# Data cleaning steps
df_clean = df.copy()

# Drop duplicate customer_ids if any
df_clean = df_clean.drop_duplicates(subset="customer_id", keep="first")

# Ensure numeric columns are correct type
numeric_cols = ["age", "tenure", "monthly_charges", "total_charges", "churn"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Drop rows with missing critical values
before = len(df_clean)
df_clean = df_clean.dropna(subset=numeric_cols + ["contract_type", "payment_method"])
after = len(df_clean)
print(f"Rows removed during cleaning: {before - after}")
print(f"Clean dataset shape: {df_clean.shape}")

## 2. Churn Rate Analysis

In [ ]:
# Overall churn rate
total = len(df_clean)
churned = df_clean["churn"].sum()
churn_rate = df_clean["churn"].mean() * 100

print(f"Total customers : {total:,}")
print(f"Churned customers: {churned:,}")
print(f"Retained customers: {total - churned:,}")
print(f"Overall churn rate : {churn_rate:.2f}%")

In [ ]:
# Churn rate by contract type
churn_by_contract = (
    df_clean.groupby("contract_type")["churn"]
    .agg(total="count", churned="sum", churn_rate="mean")
    .reset_index()
)
churn_by_contract["churn_rate_pct"] = (churn_by_contract["churn_rate"] * 100).round(2)
print("Churn Rate by Contract Type:")
print(churn_by_contract[["contract_type", "total", "churned", "churn_rate_pct"]])

# Churn rate by payment method
churn_by_payment = (
    df_clean.groupby("payment_method")["churn"]
    .agg(total="count", churned="sum", churn_rate="mean")
    .reset_index()
)
churn_by_payment["churn_rate_pct"] = (churn_by_payment["churn_rate"] * 100).round(2)
print("\nChurn Rate by Payment Method:")
print(churn_by_payment[["payment_method", "total", "churned", "churn_rate_pct"]])

## 3. Visual Analysis

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df_clean, x="churn", ax=axes[0], palette="Set2")
axes[0].set_title("Churn Distribution")
axes[0].set_xticklabels(["Retained (0)", "Churned (1)"])

churn_by_contract_plot = df_clean.groupby("contract_type")["churn"].mean().reset_index()
sns.barplot(data=churn_by_contract_plot, x="contract_type", y="churn", ax=axes[1], palette="viridis")
axes[1].set_title("Churn Rate by Contract Type")
axes[1].set_ylabel("Churn Rate")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly charges and tenure analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df_clean, x="churn", y="monthly_charges", ax=axes[0], palette="Set2")
axes[0].set_title("Monthly Charges by Churn Status")
axes[0].set_xticklabels(["Retained", "Churned"])

sns.boxplot(data=df_clean, x="churn", y="tenure", ax=axes[1], palette="Set2")
axes[1].set_title("Tenure by Churn Status")
axes[1].set_xticklabels(["Retained", "Churned"])

plt.tight_layout()
plt.show()

In [ ]:
# Payment method and age analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_payment = df_clean.groupby("payment_method")["churn"].mean().reset_index()
sns.barplot(data=churn_payment, x="payment_method", y="churn", ax=axes[0], palette="rocket")
axes[0].set_title("Churn Rate by Payment Method")
axes[0].set_ylabel("Churn Rate")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(data=df_clean, x="age", hue="churn", kde=True, ax=axes[1], palette="Set1", alpha=0.6)
axes[1].set_title("Age Distribution by Churn Status")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_features = ["age", "tenure", "monthly_charges", "total_charges", "churn"]
corr_matrix = df_clean[numeric_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for key numeric features (sample for performance)
sample_df = df_clean.sample(min(500, len(df_clean)), random_state=42)
sns.pairplot(
    sample_df[["age", "tenure", "monthly_charges", "churn"]],
    hue="churn",
    palette="Set2",
    diag_kind="kde",
    plot_kws={"alpha": 0.6, "s": 20},
)
plt.suptitle("Pairplot of Key Features", y=1.02)
plt.show()

## 4. Key Findings

- **Month-to-month** contracts have the highest churn rate.
- Customers paying via **Electronic check** tend to churn more.
- Lower **tenure** and higher **monthly charges** correlate with churn.
- Clean dataset is ready for modeling in `02_Model.ipynb`.